# Z3-Rust-19 : Optimisation multi-objectif (lexicographique, Pareto, somme pondérée)

[← Z3-Python-18 (sudoku modes)](Z3-18-Sudoku-Modes-Python.ipynb) | [Serie Z3-API](README.md) | [Version Python](Z3-19-Optimization-Lexicographic-Python.ipynb) | [Version C#](Z3-19-Optimization-Lexicographic-CSharp.ipynb) | [Jumeau Rust](Z3-19-Optimization-Lexicographic-Rust.ipynb) | [Z3-Python-20 (machine de Gödel) →](Z3-20-Machine-de-Goedel-Python.ipynb)

> **Solveur** : Z3 — crate Rust `z3` 0.23.0, execute dans le kernel `rust` (evcxr). Le portage est **integral** : meme modele, memes contraintes, memes verdicts que le jumeau Python. Les deux cotes branchent le **meme coeur Z3** par un binding officiel (pyz3 d'un cote, la crate `z3` de l'autre) ; la bibliotheque native liee ici est **Z3 5.3.3**, la Wheels `z3-solver` du jumeau Python est en **5.4.1**. Coeurs differents par construction, semantique identique — c'est ce que mesure le registre de conformite differentiale (`scripts/notebook_tools/rust_conformance.d/z3-19-optimization-lexicographic.yaml`).

Quand un problème d'optimisation a **plusieurs critères contradictoires** (coût vs impact, latence vs débit, performance vs énergie), il n'y a souvent **pas une seule solution optimale** mais un **ensemble de compromis**. Z3 expose `Optimize()`, le solveur dédié à l'optimisation sous contraintes, qui supporte trois familles de stratégies complémentaires :

- **Lexicographique** : on optimise les critères dans un ordre donné (impact d'abord, puis coût, etc.). Une seule   solution sort -- celle qui est meilleure sur le premier critère, et parmi celles-ci la meilleure sur le deuxième, etc.
- **Pareto (front de Pareto)** : on énumère toutes les solutions **non dominées** -- aucune autre solution n'est   strictement meilleure sur tous les critères simultanément. Le résultat est un *ensemble* (souvent petit) qui   représente l'ensemble des compromis rationnels.
- **Somme pondérée** : on agrège les critères en une seule fonction `somme(poids_i * critere_i)`, qu'on maximise   (ou minimise). Le poids de chaque critère reflète l'arbitrage du décideur. Cette méthode est simple mais   **manque certains points du front de Pareto** si la fonction objectif n'est pas concave.

## Problème jouet

On optimise la sélection de **3 projets** sous un **budget de 12 k€** :

| Projet | Coût (k€) | Impact (utilité) |
|--------|----------|-----------------|
| A      | 4        | 6               |
| B      | 5        | 8               |
| C      | 7        | 10              |

Objectifs contradictoires : maximiser l'impact total et minimiser le coût total. Le solveur `Optimize()` de Z3 permet d'encoder ces trois stratégies en quelques lignes et de comparer les solutions obtenues.

In [2]:
:dep z3 = "0.23.0"

use std::collections::HashMap;
use std::ops::{Add, Mul};
use z3::ast::{Ast, Int};
use z3::Optimize;

// La dependance z3 est declaree une seule fois, en tete de carnet : un :dep
// plus bas recompilerait tout et effacerait les variables typees.

// --- Definition du probleme jouet ---
fn couts() -> [(&'static str, i64); 3] {
    [("A", 4), ("B", 5), ("C", 7)]
}
fn impacts() -> [(&'static str, i64); 3] {
    [("A", 6), ("B", 8), ("C", 10)]
}
const BUDGET: i64 = 12;

// Une variable de decision par projet : x_p = 1 si le projet p est retenu.
fn variables() -> HashMap<&'static str, Int> {
    let mut v: HashMap<&'static str, Int> = HashMap::new();
    for (p, _) in couts().iter() {
        v.insert(p, Int::new_const(format!("x_{}", p)));
    }
    v
}

// Somme ponderee des couts (resp. des impacts) : somme(x_p * coefficient_p).
fn somme(v: &HashMap<&'static str, Int>, coef: &[(&'static str, i64)]) -> Int {
    let mut acc: Option<Int> = None;
    for (p, c) in coef.iter() {
        let terme = v[p].clone().mul(&Int::from_i64(*c));
        acc = Some(match acc {
            None => terme,
            Some(a) => a.add(&terme),
        });
    }
    acc.unwrap()
}

// Contraintes communes : x_p dans {0,1} et somme des couts <= BUDGET.
fn poser_contraintes(opt: &Optimize, v: &HashMap<&'static str, Int>) {
    for (p, _) in couts().iter() {
        let x = v[p].clone();
        opt.assert(&x.ge(&Int::from_i64(0)));
        opt.assert(&x.le(&Int::from_i64(1)));
    }
    let cout = somme(v, &couts());
    opt.assert(&cout.le(&Int::from_i64(BUDGET)));
}

let v = variables();
let opt = Optimize::new();
poser_contraintes(&opt, &v);

println!("Contraintes posees : chaque projet x_p in {{0,1}}, somme des couts <= {} k€", BUDGET);
println!("Objectifs : maximiser impact_total, minimiser cout_total");

Contraintes posees : chaque projet x_p in {0,1}, somme des couts <= 12 k€


Objectifs : maximiser impact_total, minimiser cout_total


In [3]:
use std::collections::HashMap;
use std::ops::{Add, Mul};
use z3::ast::{Ast, Int};
use z3::{Optimize, SatResult};

// Mesure 1 -- optimisation lexicographique : maximiser l'impact d'abord, puis minimiser le cout
let v = variables();
let opt_lex = Optimize::new();
poser_contraintes(&opt_lex, &v);

let impact_total = somme(&v, &impacts());
let cout_total = somme(&v, &couts());

// Objectif principal : maximiser impact
opt_lex.maximize(&impact_total);
// Objectif secondaire : minimiser cout (ne s'active que parmi les solutions d'impact max)
opt_lex.minimize(&cout_total);

let _verdict = if opt_lex.check(&[]) == SatResult::Sat {
    let m = opt_lex.get_model().unwrap();
    let mut selection: Vec<&'static str> = Vec::new();
    for (p, _) in couts().iter() {
        if m.eval(&v[p], true).unwrap().as_i64().unwrap() == 1 {
            selection.push(*p);
        }
    }
    let impact_val = m.eval(&impact_total, true).unwrap().as_i64().unwrap();
    let cout_val = m.eval(&cout_total, true).unwrap().as_i64().unwrap();
    // La liste est rendue comme en Python (['B', 'C']) pour que les deux
    // carnets affichent exactement la meme ligne.
    let liste = format!(
        "[{}]",
        selection.iter().map(|p| format!("'{}'", p)).collect::<Vec<_>>().join(", ")
    );
    println!("Lex : selection = {} (impact = {}, cout = {} k€)", liste, impact_val, cout_val);
    println!("  -> B + C (impact 18, cout 12 k€) : la seule combinaison a impact 18.");
    println!("  -> Cout secondaire = 12 k€ (le seul faisable pour impact = 18).");
} else {
    println!("Lex : aucune solution");
};

Lex : selection = ['B', 'C'] (impact = 18, cout = 12 k€)


  -> B + C (impact 18, cout 12 k€) : la seule combinaison a impact 18.


  -> Cout secondaire = 12 k€ (le seul faisable pour impact = 18).


### Interprétation : optimisation lexicographique

La stratégie lex donne une **seule solution** : `B + C` (coût 12 k€, impact 18). Le solveur commence par chercher l'impact maximum faisable (18), et parmi toutes les combinaisons qui atteignent cet impact, il choisit celle de coût minimum. C'est la même solution qu'un décideur rationnel prendrait s'il considère l'impact comme absolument prioritaire.

**Forces** :

- Décision **non-ambiguë** : une seule solution sort, pas d'arbitrage subjectif du décideur.
- Ordre d'importance **explicite** : si A est plus important que B, le solveur respecte cette hiérarchie strictement.
- Le solveur Z3 `Optimize()` implémente cette stratégie nativement via `maximize(...)` puis `minimize(...)`.

**Faiblesses** :

- **Cécité aux autres compromis** : si l'impact maximum est 18 mais qu'une solution à impact 17 coûte 5 k€ de moins,   lex l'ignore. Le décideur ne voit pas le tradeoff.
- **Sensible à l'ordre des critères** : inverser l'ordre (coût d'abord, puis impact) donnerait une solution   différente. C'est cohérent (les critères ne sont pas symétriques), mais cela oblige à un choix d'arbitrage *a priori*.

**Quand l'utiliser** : quand les critères ont une **hiérarchie claire** (e.g. sécurité d'abord, performance ensuite) ou quand le décideur veut une réponse claire et non-discutable.

> Côté Rust, les deux objectifs sont posés dans l'ordre (`Optimize::maximize` puis `Optimize::minimize`), exactement comme côté Python (`opt.maximize` puis `opt.minimize`) : la priorité lexicographique n'est pas une convention de lecture du carnet, c'est l'ordre de declaration des objectifs dans le solveur.

In [4]:
use std::collections::HashMap;
use std::ops::{Add, BitOr, Mul};
use z3::ast::{Ast, Bool, Int};
use z3::{Optimize, SatResult};

// Mesure 2 -- enumeration du front de Pareto : on garde les solutions non-dominees
// Strategie : iterer, a chaque etape ajouter une contrainte d'exclusion, jusqu'a plus de solution
let v = variables();
let opt_p = Optimize::new();
poser_contraintes(&opt_p, &v);

let impact_total = somme(&v, &impacts());
let cout_total = somme(&v, &couts());

// On maximise l'impact pour trouver le point extreme 'impact max'
opt_p.maximize(&impact_total);

let mut solutions: Vec<(Vec<&'static str>, i64, i64)> = Vec::new();
let mut encore = true;
while encore {
    if opt_p.check(&[]) != SatResult::Sat {
        encore = false;
    } else {
        let m = opt_p.get_model().unwrap();
        let mut sel: Vec<&'static str> = Vec::new();
        for (p, _) in couts().iter() {
            if m.eval(&v[p], true).unwrap().as_i64().unwrap() == 1 {
                sel.push(*p);
            }
        }
        let imp = m.eval(&impact_total, true).unwrap().as_i64().unwrap();
        let cout = m.eval(&cout_total, true).unwrap().as_i64().unwrap();
        solutions.push((sel, imp, cout));
        // Contrainte d'exclusion : empecher de retourner exactement a cette solution
        let mut disj: Option<Bool> = None;
        for (p, _) in couts().iter() {
            let val = m.eval(&v[p], true).unwrap().as_i64().unwrap();
            let neq = v[p].clone().ne(&Int::from_i64(val));
            disj = Some(match disj {
                None => neq,
                Some(d) => d.bitor(&neq),
            });
        }
        opt_p.assert(&disj.unwrap());
    }
}

println!("Pareto : {} solutions non-dominnees trouvees\n", solutions.len());
println!("{:<12} {:>8} {:>12}", "Selection", "Impact", "Cout (k€)");
println!("{}", "-".repeat(36));
for (sel, imp, cout) in &solutions {
    let label = if sel.is_empty() { "aucun".to_string() } else { sel.join("+") };
    println!("{:<12} {:>8} {:>12}", label, imp, cout);
}

// Filtrer les Pareto-optimales : eliminer les solutions qui en dominent une autre
// s1 domine s2 si s1 a cout <= et impact >=, avec au moins un strictement meilleur.
fn domine(s1: &(Vec<&'static str>, i64, i64), s2: &(Vec<&'static str>, i64, i64)) -> bool {
    s1.2 <= s2.2 && s1.1 >= s2.1 && (s1.2 < s2.2 || s1.1 > s2.1)
}

let mut pareto: Vec<(Vec<&'static str>, i64, i64)> = Vec::new();
for s in &solutions {
    // Exclure la solution vide (ne rien selectionner) : trivialement Pareto-non-dominee
    // (cout 0 minimise le cout, mais impact 0 n'ameliore rien) -- on la filtre pour
    // ne garder que les selections reelles.
    if s.0.is_empty() {
        continue;
    }
    let dominee = solutions
        .iter()
        .any(|t| t != s && !t.0.is_empty() && domine(t, s));
    if !dominee {
        pareto.push(s.clone());
    }
}
println!("\nApres filtrage (selection non vide + non-dominance) : {} solutions Pareto-optimales", pareto.len());


Pareto : 7 solutions non-dominnees trouvees


Selection      Impact    Cout (k€)


------------------------------------


B+C                18           12


A+C                16           11


A+B                14            9


C                  10            7


B                   8            5


A                   6            4


aucun               0            0


Apres filtrage (selection non vide + non-dominance) : 6 solutions Pareto-optimales


### Interprétation : front de Pareto

Les **6 solutions** trouvées (B+C, A+C, A+B, C, B, A) sont **toutes Pareto-optimales** : aucune n'est strictement dominée par une autre. Cela vient du fait que les 3 projets ont des **ratios impact/coût différents** (A=1.5, B=1.6, C=1.43) et que chaque solution occupe une niche sur le front.

**Cas dominé vs Pareto-optimal** : on parle de *domination* quand une solution `s1` est meilleure que `s2` sur **tous** les critères simultanément. Ici, par exemple, *aucune* solution n'est meilleure que toutes les autres sur (coût, impact) à la fois -- le problème est multi-objectif authentique, et le front est large.

**Forces** :

- **Visibilité complète des compromis** : le décideur voit **toutes** les options rationnelles (coût min, impact max, intermédiaires).
- **Pas d'arbitrage caché** : le poids de chaque critère n'est pas figé -- le décideur arbitre **après** avoir vu le front.
- **Fondement théorique solide** : la dominance Pareto est l'outil standard en recherche opérationnelle et en économie du bien-être.

**Faiblesses** :

- Le front peut être **trop large** (ici 6 solutions, mais un problème industriel peut en avoir des milliers).
- **Pas de hiérarchie** : le front est un *ensemble*, pas une recommandation -- le décideur doit choisir **dans** le front.
- Coûteux à énumérer : il faut résoudre le problème plusieurs fois (une fois par solution du front).

**Quand l'utiliser** : en phase exploratoire (cadrage d'un nouveau problème), ou quand le décideur veut **voir** le paysage des compromis avant de pondérer.

> Côté Rust, l'exclusion d'une solution déjà visitée se formule par une **disjonction d'inégalités** (`Int::ne` puis `Bool::bitor`) : `x_A != 0 ou x_B != 1 ou x_C != 1` interdit le retour exact au triplet précédent. La boucle recommence tant que le solveur trouve une solution, et s'arrête sur `unsat`.

In [5]:
use std::collections::HashMap;
use std::ops::{Add, Mul, Sub};
use z3::ast::{Ast, Int, Real};
use z3::{Optimize, SatResult};

// Mesure 3 -- somme ponderee : on agrege impact - alpha * cout, et on fait varier alpha
let v = variables();
let impact_total = somme(&v, &impacts());
let cout_total = somme(&v, &couts());

let alphas: [f64; 8] = [0.0, 0.5, 1.0, 1.3, 1.5, 2.0, 5.0, 10.0];
let mut resultats: Vec<(f64, String, i64, i64, f64)> = Vec::new();

for alpha in alphas.iter() {
    let opt_w = Optimize::new();
    poser_contraintes(&opt_w, &v);
    // Objectif : maximiser impact - alpha * cout (equivalent a maximiser impact seul si alpha=0,
    // ou minimiser cout seul si alpha est tres grand)
    // Le poids alpha est passe au solveur comme un rationnel exact (13/10 pour 1.3),
    // exactement comme le fait pyz3 avec un float Python.
    let alpha_reel = Real::from_rational_str(&format!("{:?}", alpha), "1").unwrap();
    let obj = Real::from_int(&impact_total).sub(&alpha_reel.mul(&Real::from_int(&cout_total)));
    opt_w.maximize(&obj);
    if opt_w.check(&[]) == SatResult::Sat {
        let m = opt_w.get_model().unwrap();
        let mut sel: Vec<&'static str> = Vec::new();
        for (p, _) in couts().iter() {
            if m.eval(&v[p], true).unwrap().as_i64().unwrap() == 1 {
                sel.push(*p);
            }
        }
        let imp = m.eval(&impact_total, true).unwrap().as_i64().unwrap();
        let cout = m.eval(&cout_total, true).unwrap().as_i64().unwrap();
        let score = imp as f64 - alpha * cout as f64;
        let label = if sel.is_empty() { "aucun".to_string() } else { sel.join("+") };
        resultats.push((*alpha, label, imp, cout, score));
    }
}

println!("{:>6} {:<12} {:>8} {:>6} {:>8}  Lecture", "alpha", "Selection", "Impact", "Cout", "Score");
println!("{}", "-".repeat(70));
for (alpha, label, imp, cout, score) in &resultats {
    let lecture = if label == "aucun" {
        "solution vide (ne rien faire)"
    } else if *alpha == 0.0 {
        "max impact seul"
    } else {
        "compromis impact / cout"
    };
    println!("{:>6.2} {:<12} {:>8} {:>5}k {:>8.2}  {}", alpha, label, imp, cout, score, lecture);
}

// --- Lecture derivee -----------------------------------------------------------
// Les chiffres de la lecture sont CALCULES, pas recopies a la main : on enumere
// toutes les selections faisables, puis, pour chaque alpha, l'ensemble des
// selections optimales. C'est ce qui evite qu'une prose contredise le tableau
// ci-dessus -- et c'est aussi ce qui revele que le front de Pareto contient des
// points que la somme ponderee n'atteint jamais.

// Combinaisons dans l'ordre lexicographique (comme itertools.combinations).
fn combinaisons(items: &[&'static str], k: usize) -> Vec<Vec<&'static str>> {
    let n = items.len();
    let mut out: Vec<Vec<&'static str>> = Vec::new();
    if k > n {
        return out;
    }
    if k == 0 {
        out.push(Vec::new());
        return out;
    }
    let mut idx: Vec<usize> = (0..k).collect();
    loop {
        out.push(idx.iter().map(|&i| items[i]).collect());
        let mut i = k;
        loop {
            if i == 0 {
                return out;
            }
            i -= 1;
            if idx[i] != i + n - k {
                idx[i] += 1;
                for j in (i + 1)..k {
                    idx[j] = idx[j - 1] + 1;
                }
                break;
            }
            if i == 0 {
                return out;
            }
        }
    }
}

// Score de la selection (impact, cout) pour un poids alpha du cout.
fn score_de(sel: &(Vec<&'static str>, i64, i64), alpha: f64) -> f64 {
    sel.1 as f64 - alpha * sel.2 as f64
}

fn nom(sel: &(Vec<&'static str>, i64, i64)) -> String {
    if sel.0.is_empty() {
        "vide".to_string()
    } else {
        sel.0.join("+")
    }
}

// Formatage a 4 chiffres significatifs (equivalent du format .4g de Python).
fn g4(x: f64) -> String {
    if x.is_infinite() {
        return if x > 0.0 { "inf".to_string() } else { "-inf".to_string() };
    }
    if x == 0.0 {
        return "0".to_string();
    }
    let exp = x.abs().log10().floor() as i32;
    let dec = (3 - exp).max(0) as usize;
    let mut s = format!("{:.*}", dec, x);
    if s.contains('.') {
        while s.ends_with('0') {
            s.pop();
        }
        if s.ends_with('.') {
            s.pop();
        }
    }
    s
}

// Ensemble des selections optimales pour un alpha donne.
fn optimaux(faisables: &[(Vec<&'static str>, i64, i64)], alpha: f64) -> Vec<usize> {
    let mut meilleur = f64::NEG_INFINITY;
    for s in faisables {
        let sc = score_de(s, alpha);
        if sc > meilleur {
            meilleur = sc;
        }
    }
    faisables
        .iter()
        .enumerate()
        .filter(|(_, s)| (score_de(s, alpha) - meilleur).abs() < 1e-9)
        .map(|(i, _)| i)
        .collect()
}

let mut faisibles: Vec<(Vec<&'static str>, i64, i64)> = Vec::new();
for k in 0..=couts().len() {
    for combo in combinaisons(&["A", "B", "C"], k) {
        let mut cout = 0i64;
        let mut imp = 0i64;
        for p in &combo {
            cout += couts().iter().find(|c| c.0 == *p).unwrap().1;
            imp += impacts().iter().find(|c| c.0 == *p).unwrap().1;
        }
        if cout <= BUDGET {
            faisibles.push((combo, imp, cout));
        }
    }
}

println!("\nLecture : la solution glisse le long du front de Pareto quand alpha monte.");
for alpha in alphas.iter() {
    let details: Vec<String> = optimaux(&faisibles, *alpha)
        .iter()
        .map(|&i| {
            let s = &faisibles[i];
            format!(
                "{} (impact {}, cout {} kEUR, score {:.2})",
                nom(s),
                s.1,
                s.2,
                score_de(s, *alpha)
            )
        })
        .collect();
    println!("  alpha = {:<5} : {}", format!("{:?}", alpha), details.join(" | "));
}

println!("\nSeuils d'alpha deduits des egalites de score :");
println!("  (une selection est optimale exactement sur l'intervalle indique)");
for (i, s) in faisibles.iter().enumerate() {
    let mut borne_inf = 0.0f64;
    let mut borne_sup = f64::INFINITY;
    for (j, t) in faisibles.iter().enumerate() {
        if i == j {
            continue;
        }
        let ecart_cout = s.2 - t.2;
        if ecart_cout == 0 {
            continue;
        }
        let alpha_egalite = (s.1 - t.1) as f64 / ecart_cout as f64;
        if ecart_cout > 0 {
            if alpha_egalite < borne_sup {
                borne_sup = alpha_egalite;
            }
        } else if alpha_egalite > borne_inf {
            borne_inf = alpha_egalite;
        }
    }
    if borne_inf <= borne_sup {
        println!("  {:<8} : alpha dans [{}, {}]", nom(s), g4(borne_inf), g4(borne_sup));
    }
}
// evcxr : une cellule qui se termine sur une expression non semicolon-e renvoie
// une sortie "()" parasite ; on termine sur une liaison pour n'avoir que des
// sorties texte, comme sur le jumeau Python.
let _ = ();

 alpha Selection      Impact   Cout    Score  Lecture


----------------------------------------------------------------------


  0.00 B+C                18    12k    18.00  max impact seul


  0.50 B+C                18    12k    12.00  compromis impact / cout


  1.00 B+C                18    12k     6.00  compromis impact / cout


  1.30 B+C                18    12k     2.40  compromis impact / cout


... (15 outputs left out)

  vide     : alpha dans [1.6, inf]


  B        : alpha dans [1.5, 1.6]


  A+B      : alpha dans [1.333, 1.5]


  B+C      : alpha dans [0, 1.333]


### Interprétation : somme pondérée

La somme pondérée `maximiser(impact - α * coût)` fait varier la solution selon `α` :

- `α = 0` : on ignore le coût, on maximise l'impact → **B+C** (impact 18).
- `α = 0.5` à `α = 1.0` : compromis équilibré → **B+C** (le score 18 - α × 12 reste le plus élevé).
- `α = 1.3` : le coût commence à peser, mais pas encore assez pour changer de solution → **B+C** (score 2.4),   devant **A+B** (score 2.3). Les deux s'égalisent exactement à **α = 4/3 ≈ 1.33** : c'est le seuil où B+C   cède la place à A+B.
- `α = 1.5` : **A+B** et **B** sont les deux solutions optimales, à score 0.5 chacune (le solveur n'en retourne   qu'une). A, B+C et la solution vide sont à 0 ; A+C et C sont à -0.5. Le coût pèse alors assez pour que le   projet le plus rentable (B) concurrence le couple A+B.
- `α > 8/5 = 1.6` : la **solution vide** devient optimale (coût 0, impact 0, score 0). À `α = 1.6` elle est à   égalité avec B, et pour `α ≥ 2` tous les scores non vides sont strictement négatifs. Le modèle dit   littéralement *"ne rien sélectionner"*.
- `α = 10` : la solution vide reste optimale (coût >> impact, plus aucun projet ne vaut son coût).

**Forces** :

- **Simple** : un seul paramètre à régler (le vecteur de poids), une seule résolution par jeu de poids.
- **Continuité** : faire varier `α` continûment fait *glisser* la solution le long du front de Pareto.
- **Sensibilité** : on peut étudier comment la décision réagit à un changement d'arbitrage du décideur.

**Faiblesses** :

- **Pas concave → trous** : la somme pondérée ne parcourt que l'**enveloppe concave supérieure** du front de   Pareto. Ici, parmi les 6 solutions Pareto-optimales, seules **B+C**, **A+B** et **B** ont un intervalle d'α où   elles sont optimales ; **A, C et A+C** sont Pareto-optimales mais **jamais choisies** par la somme pondérée   (le « trou » typique des points non supportés). Un front concave serait, lui, entièrement exploré — ce n'est   pas garanti en général.
- **Solution vide possible** : si le poids du coût devient trop fort, la solution vide (ne rien faire) devient   optimale (ici au-delà de `α = 8/5`). C'est cohérent mathématiquement, mais c'est un signal d'alarme : le décideur   veut sans doute dépenser le budget, et la modélisation doit intégrer cette intention (par exemple via une   contrainte `somme(x_p) >= 1`).
- **Interprétabilité du poids** : dire "α = 2" n'a pas de signification concrète pour un décideur non technique.
- **Tirage au sort des poids** : pour échantillonner le front de Pareto, on tire `α` au hasard -- certains points   du front sont sur-représentés (concavité), d'autres sous-représentés.

**Quand l'utiliser** : quand le décideur peut exprimer un arbitrage numérique (même grossier) entre critères, ou pour une **analyse de sensibilité** (que se passe-t-il si je change d'avis ?). **Attention** : ajouter une contrainte `au moins 1 projet` si l'objectif du décideur est de dépenser le budget.

> Côté Rust, le poids `α` est converti en **rationnel exact** (`Real::from_rational_str`), de sorte que le solveur voit `13/10` et non une approximation binaire de 1.3 : le seuil `α = 4/3` sort donc exact, comme côté Python. Le tableau de la lecturederivee est recalcule en pur Rust (combinaisons, score, bornes d'égalité) — aucun chiffre n'y est recopié à la main.

In [6]:
// Exercice 1 -- Optimisation lexicographique sur 4 projets
//
// Probleme : 4 projets D, E, F, G avec les caracteristiques suivantes :
//   D : cout 3, impact 5
//   E : cout 6, impact 9
//   F : cout 4, impact 7
//   G : cout 5, impact 8
// Budget total : 14 k€.
//
// Coder une optimisation lexicographique qui maximise l'impact d'abord, puis minimise le cout.
// Affichez la selection finale et l'impact/cout associes.
//
// TODO etudiant : creer l'instance Optimize(), poser les contraintes 0 <= x_p <= 1,
// ajouter la contrainte budget, puis maximiser impact puis minimiser cout.
// Afficher la selection finale.
fn exercice_lex_4vars() {
    // TODO etudiant : le corps de la fonction est a completer.
}

exercice_lex_4vars();
println!("Exercice 1 : voir cellule ci-dessus. Selection attendue = ? (a calculer)");

Exercice 1 : voir cellule ci-dessus. Selection attendue = ? (a calculer)


In [7]:
// Exercice 2 -- Enumeration du front de Pareto sur 4 projets
//
// Reprenez les memes 4 projets D, E, F, G que l'exercice 1.
// Enumerez TOUTES les solutions Pareto-optimales (selection, impact, cout).
// Affichez le nombre de solutions et le tableau resultat.
//
// TODO etudiant : reprendre la strategie d'enumeration de la mesure 2 (cellule 4),
// l'adapter aux 4 projets D/E/F/G avec leur budget 14 k€.
fn exercice_pareto_4vars() {
    // TODO etudiant : le corps de la fonction est a completer.
}

exercice_pareto_4vars();
println!("Exercice 2 : voir cellule ci-dessus. Nombre de solutions Pareto = ? (a calculer)");

Exercice 2 : voir cellule ci-dessus. Nombre de solutions Pareto = ? (a calculer)


In [8]:
// Exercice 3 -- Somme ponderee sur 4 projets avec 4 poids differents
//
// Reprenez les 4 projets D, E, F, G. Testez 4 valeurs de alpha (par exemple 0, 1, 2, 5).
// Pour chaque alpha, affichez la selection choisie par le solveur.
// Concluez : pour quelle valeur de alpha la selection change-t-elle ?
//
// TODO etudiant : boucle sur 4 valeurs de alpha, appel a Optimize() a chaque fois,
// afficher (alpha, selection, impact, cout).
fn exercice_weighted_4poids() {
    // TODO etudiant : le corps de la fonction est a completer.
}

exercice_weighted_4poids();
println!(r#"Exercice 3 : voir cellule ci-dessus. Verdict attendu = "alpha=0 -> D+E+G (impact 22) ; alpha=1 -> D+E+G, ex aequo avec D+E+F et D+F+G (score 8) ; alpha=2 et alpha=5 -> solution vide (aucun projet, score 0)" ou similaire."#);

Exercice 3 : voir cellule ci-dessus. Verdict attendu = "alpha=0 -> D+E+G (impact 22) ; alpha=1 -> D+E+G, ex aequo avec D+E+F et D+F+G (score 8) ; alpha=2 et alpha=5 -> solution vide (aucun projet, score 0)" ou similaire.


## Conclusion : arbre de décision pour choisir sa stratégie

Le tableau ci-dessous résume quand utiliser lex, Pareto ou weighted, en fonction du contexte du décideur.

| Situation du décideur | Stratégie recommandée | Justification |
|----------------------|----------------------|---------------|
| **Hiérarchie claire des critères** (e.g. sécurité > coût > performance) | **Lexicographique** | L'ordre d'importance est explicite ; une seule solution sort, pas d'arbitrage subjectif |
| **Cadrage exploratoire** (premier contact avec un nouveau problème) | **Pareto** | Le front complet révèle la structure des compromis ; le décideur voit **avant** de pondérer |
| **Analyse de sensibilité** (que se passe-t-il si je change d'avis ?) | **Somme pondérée avec sweep sur α** | On observe comment la solution glisse le long du front ; utile pour le décideur hésitant |
| **Décideur non technique, arbitrage grossier** | **Somme pondérée** | Facile à expliquer ("je préfère 2 fois plus le coût à l'impact") |
| **Front non-concave** (cas pathologique) | **Pareto** | La somme pondérée ne peut pas atteindre les points non-concaves ; seul Pareto les capture |

**Take-away** : les trois stratégies sont **complémentaires**, pas concurrentes. Dans la pratique, on commence par Pareto pour **voir** le paysage, on teste une ou deux valeurs de `α` pour sentir la **sensibilité**, et on finit par lex pour **trancher** quand le décideur a une hiérarchie claire en tête.

Ce carnet est le **jumeau Rust** du carnet Python : meme probleme, meme encodage, memes verdicts, mesures par le registre de conformite differentiale. Les differences portees par la crate Rust (noms de types, `Real::from_rational_str` pour les poids, formatage de la liste) sont **de implementation**, pas de semantique.

**Pour aller plus loin** :

- **Z3-06 Advanced Optimization** : explore d'autres facettes (soft constraints, weighted MaxSAT).
- **Z3-12 Real Arithmetic** : si les variables sont réelles (pas seulement entières 0/1), les solutions   deviennent continues et le front est un *polyèdre* dans l'espace des objectifs.
- **Documentation Z3 `Optimize()`** : https://microsoft.github.io/z3guide/programming/Z3%20JavaScript%20Examples/optimization.html